# Auditoria de Licitações e Contratos (PNCP e MIDES)
Este notebook reproduz os resultados apresentados no artigo, cruzando os microdados do Portal Nacional de Contratações Públicas (PNCP) e Municípios em Destaque (MIDES) com o Cadastro de Empresas Inidôneas e Suspensas (CEIS).

In [ ]:
import pandas as pd
import glob
import warnings
warnings.filterwarnings('ignore')

TIPOS_IMPEDITIVOS = [
    'Impedimento/proibição de contratar com prazo determinado',
    'Impedimento/proibição de contratar sem prazo determinado',
    'Suspensão',
    'Declaração de Inidoneidade sem prazo determinado',
    'Declaração de Inidoneidade com prazo determinado',
    'Suspensão/Interdição das atividades sem prazo determinado',
]

print("Bibliotecas carregadas.")

: 

## 1. Análise de Anomalias no PNCP (Federal)

In [ ]:
# 1. Carregar Licitações e Contratos
compras_files = glob.glob("../../etl/data/pncp/pncp_2021*.json")
df_pncp_compras = pd.concat([pd.read_json(f) for f in compras_files], ignore_index=True)

contratos_files = glob.glob("../../etl/data/pncp_contratos/pncp_2021*.json")
df_pncp_contratos = pd.concat([pd.read_json(f) for f in contratos_files], ignore_index=True)

# 2. Carregar CEIS
df_sancoes = pd.read_csv('../../etl/data/sanctions/ceis.csv', sep=',', encoding='latin-1', on_bad_lines='skip', low_memory=False)
cpf_cnpj_col = [c for c in df_sancoes.columns if 'CPF' in c.upper() or 'CNPJ' in c.upper()][0]
df_sancoes['cnpj_limpo'] = df_sancoes[cpf_cnpj_col].astype(str).str.replace(r'\D', '', regex=True).str.zfill(14)
df_sancoes['data_inicio_dt'] = pd.to_datetime(df_sancoes['data_inicio'], format='%d/%m/%Y', errors='coerce')
df_sancoes['data_fim_dt'] = pd.to_datetime(df_sancoes['data_fim'], format='%d/%m/%Y', errors='coerce')
df_sancoes_imp = df_sancoes[df_sancoes['tipo_sancao'].isin(TIPOS_IMPEDITIVOS)]

# 3. Cruzamento
df_pncp_compras['numero_controle_compra'] = df_pncp_compras['numeroControlePNCP']
df_pncp_contratos['numero_controle_compra'] = df_pncp_contratos['numeroControlePncpCompra']
df_pncp_contratos['ni_limpo'] = df_pncp_contratos['niFornecedor'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(14)

df_cruzado_pncp = pd.merge(df_pncp_compras, df_pncp_contratos, on='numero_controle_compra', suffixes=('_compra', '_contrato'))
df_cruzado_pncp['valor_estimado'] = pd.to_numeric(df_cruzado_pncp['valorTotalEstimado'], errors='coerce')
df_cruzado_pncp['valor_contratado'] = pd.to_numeric(df_cruzado_pncp['valorInicial'], errors='coerce')
df_cruzado_pncp['data_abertura'] = pd.to_datetime(df_cruzado_pncp['dataAberturaProposta'], errors='coerce')
df_cruzado_pncp['data_assinatura'] = pd.to_datetime(df_cruzado_pncp['dataAssinatura'], errors='coerce')
df_cruzado_pncp['modalidade_id'] = pd.to_numeric(df_cruzado_pncp['modalidadeId'], errors='coerce')

# Cálculos
cronologia_reversa = df_cruzado_pncp[df_cruzado_pncp['data_assinatura'] < df_cruzado_pncp['data_abertura']]
aditivo_ilegal = df_cruzado_pncp[df_cruzado_pncp['valor_contratado'] > (df_cruzado_pncp['valor_estimado'] * 1.25)]
fuga_modalidade_pncp = df_cruzado_pncp[(df_cruzado_pncp['modalidade_id'] == 8.0) & (df_cruzado_pncp['valor_contratado'] > 100000.00)]

merged_ceis = pd.merge(df_pncp_contratos, df_sancoes_imp, left_on='ni_limpo', right_on='cnpj_limpo')
merged_ceis['data_ass_dt'] = pd.to_datetime(merged_ceis['dataAssinatura'], errors='coerce')
fraude_ceis = merged_ceis[
    (merged_ceis['data_inicio_dt'] <= merged_ceis['data_ass_dt']) &
    (merged_ceis['data_fim_dt'].isna() | (merged_ceis['data_fim_dt'] >= merged_ceis['data_ass_dt']))
]
fraude_ceis = fraude_ceis.drop_duplicates(subset=['numeroControlePNCP', 'ni_limpo'])

print(f"Cronologia Reversa (PNCP): {len(cronologia_reversa)}")
print(f"Distorções Financeiras (PNCP): {len(aditivo_ilegal)}")
print(f"Fuga de Modalidade (PNCP): {len(fuga_modalidade_pncp)}")
print(f"Sanção Vigente (PNCP): {len(fraude_ceis)}")


## 2. Análise de Anomalias no MIDES (Municipal)

In [ ]:
MIDES_DIR = '../../etl/data/mides'
df_mides_lic = pd.read_csv(f'{MIDES_DIR}/licitacao.csv', low_memory=False)
df_mides_ctr = pd.read_csv(f'{MIDES_DIR}/contrato.csv', low_memory=False)

df_lic_unique = df_mides_lic[['licitacao_id', 'estado', 'modalidade', 'valor_estimado', 'data_publicacao']].drop_duplicates(subset=['licitacao_id'])
df_ctr_unique = df_mides_ctr[['licitacao_id', 'valor', 'cnpj_fornecedor', 'data_assinatura']].drop_duplicates(subset=['licitacao_id', 'cnpj_fornecedor'])

df_lic_unique['valor_estimado'] = pd.to_numeric(df_lic_unique['valor_estimado'], errors='coerce')
df_ctr_unique['valor'] = pd.to_numeric(df_ctr_unique['valor'], errors='coerce')
df_lic_unique['data_publicacao'] = pd.to_datetime(df_lic_unique['data_publicacao'], errors='coerce')
df_ctr_unique['data_assinatura'] = pd.to_datetime(df_ctr_unique['data_assinatura'], errors='coerce')

df_audit = pd.merge(df_lic_unique, df_ctr_unique, on='licitacao_id', how='inner')
df_audit = df_audit.dropna(subset=['valor_estimado', 'valor'])
df_audit_fin = df_audit[(df_audit['valor_estimado'] > 1000) & (df_audit['valor'] > 1000) & (df_audit['valor'] < 50e9)].copy()

# Cálculos
df_crono = df_audit.dropna(subset=['data_publicacao', 'data_assinatura'])
fraude_temporal = df_crono[df_crono['data_assinatura'] < df_crono['data_publicacao']]

df_audit_fin['sobrepreco_pct'] = ((df_audit_fin['valor'] - df_audit_fin['valor_estimado']) / df_audit_fin['valor_estimado'] * 100)
aditivos_ilegais = df_audit_fin[(df_audit_fin['sobrepreco_pct'] > 25) & (df_audit_fin['sobrepreco_pct'] <= 200) & (df_audit_fin['valor'] > 10000)]

def limite_dispensa(ano):
    if pd.isna(ano) or ano <= 2018: return 15000.00
    elif ano == 2019: return 33000.00
    elif ano in [2020, 2021]: return 100000.00
    elif ano == 2022: return 108041.15
    elif ano == 2023: return 114416.65
    elif ano == 2024: return 119812.02
    elif ano == 2025: return 125451.15
    else: return 130984.20

df_audit['ano_assinatura'] = df_audit['data_assinatura'].dt.year
df_audit['teto_dispensa'] = df_audit['ano_assinatura'].apply(limite_dispensa)
dispensas_irregulares = df_audit[(df_audit['modalidade'] == 8.0) & (df_audit['valor'] > df_audit['teto_dispensa'])]

df_ctr_unique['cnpj_fornecedor'] = df_ctr_unique['cnpj_fornecedor'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(14)
df_audit_ceis = pd.merge(df_ctr_unique, df_sancoes_imp, left_on='cnpj_fornecedor', right_on='cnpj_limpo', how='inner')
fraude_ceis_ativa = df_audit_ceis[
    (df_audit_ceis['data_assinatura'] >= df_audit_ceis['data_inicio_dt']) &
    ((df_audit_ceis['data_assinatura'] <= df_audit_ceis['data_fim_dt']) | df_audit_ceis['data_fim_dt'].isna())
]
fraude_ceis_ativa = fraude_ceis_ativa.drop_duplicates(subset=['licitacao_id', 'cnpj_fornecedor'])

print(f"Cronologia Reversa (MIDES): {len(fraude_temporal)}")
print(f"Distorções Financeiras (MIDES): {len(aditivos_ilegais)}")
print(f"Fuga de Modalidade (MIDES): {len(dispensas_irregulares)}")
print(f"Sanção Vigente (MIDES): {len(fraude_ceis_ativa)}")


---
## 3. Nota Metodológica de Auditoria: Trade-off de Ingestão TSE vs. Contratos Públicos
> **Reflexão Arquitetural para o TCC:** Ao cruzar dados de doações de campanha (TSE) com compras públicas (PNCP), a abordagem *Match-First* (filtrar apenas quem já é sócio de fornecedor com contrato) assegura altíssima performance computacional, mas impõe limitações analíticas importantes:
> 1. **Autopagamento de Campanha:** Não detecta candidatos contratando a própria empresa privada com o Fundo Eleitoral se essa empresa não tiver contratos no PNCP.
> 2. **Ciclos Eleitorais Fechados:** Não captura triangulações de doações partidárias que operam fora da administração direta.
> 3. **Dados Assíncronos (*Late-Arriving Data*):** Novos contratos baixados em background exigem reprocessamento do match.
>
> **Proposta Metodológica de Evolução:** Substituir o filtro por correspondência contratual pelo **Filtro por Relevância Financeira Global (Corte por valor >= R$ 1.000)**, reduzindo o ruído sem perder os casos intra-campanha.
